# 🎯 МИДТЕРМ 2 — Сурет Классификациясы
## Neural Network / Deep Learning жобасы

**Датасет:** CIFAR-10 (Kaggle / open source)  
**Датасет сілтемесі:** https://www.cs.toronto.edu/~kriz/cifar.html  
**Тапсырма:** 10 санатқа суреттерді жіктеу (самолет, автомобиль, құс, мысық және т.б.)  
**Модельдер:** 1) Қарапайым ANN, 2) CNN (Convolutional Neural Network)


## 1. Кітапханаларды жүктеу

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import cifar10

from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow нұсқасы:', tf.__version__)
print('Барлық кітапхана сәтті жүктелді ✅')

## 2. Деректерді жүктеу (Dataset)

CIFAR-10 — ашық датасет. 60,000 сурет, 10 санат:  
`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`

In [ ]:
# Деректерді жүктеу
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Санат атаулары
class_names = ['Самолет', 'Автомобиль', 'Құс', 'Мысық', 'Бұғы',
               'Ит', 'Бақа', 'Жылқы', 'Кеме', 'Жүк машина']

print('Train деректері:', X_train.shape)
print('Test деректері:', X_test.shape)
print('Сурет өлшемі: 32x32 пиксель, 3 түс (RGB)')

## 3. EDA — Деректерді зерттеу (Exploratory Data Analysis)

In [ ]:
# 3.1 Деректер туралы жалпы ақпарат
print('=== ДЕРЕКТЕР ТУРАЛЫ АҚПАРАТ ===')
print(f'Train санаты: {X_train.shape[0]} сурет')
print(f'Test санаты:  {X_test.shape[0]} сурет')
print(f'Пиксель мәні ауқымы: {X_train.min()} - {X_train.max()}')
print(f'Санаттар саны: {len(class_names)}')

In [ ]:
# 3.2 Кейбір суреттерді көрсету
plt.figure(figsize=(12, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis('off')
plt.suptitle('CIFAR-10 датасетінен үлгілер', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Санаттар бойынша бөлінуі
unique, counts = np.unique(y_train, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(class_names, counts, color='steelblue')
plt.title('Train деректеріндегі санаттар бойынша бөліну')
plt.xlabel('Санат')
plt.ylabel('Сурет саны')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
print('Деректер теңдестірілген (balanced) — әр санатта 5000 сурет')

## 4. Деректерді алдын ала өңдеу (Preprocessing)

In [ ]:
# Нормализация: пиксель мәндерін 0-255 аралығынан 0-1 аралығына түрлендіру
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

# y мәндерін жалпақтату (flatten)
y_train = y_train.flatten()
y_test  = y_test.flatten()

print('Нормализация аяқталды ✅')
print(f'Пиксель мәні ауқымы (нормализациядан кейін): {X_train.min():.1f} - {X_train.max():.1f}')

## 5. Модель 1 — Қарапайым ANN (Жасанды нейрондық желі)

Бірінші модель ретінде қарапайым толық байланысты (fully connected) желіні қолданамыз.

In [ ]:
# ANN үшін суреттерді жалпақтату (32x32x3 → 3072)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat  = X_test.reshape(X_test.shape[0], -1)

# Модельді құру
model_ann = keras.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')  # 10 санат
], name='ANN_Model')

model_ann.summary()

In [ ]:
# Компиляция
model_ann.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Оқыту
print('ANN моделін оқыту басталды...')
history_ann = model_ann.fit(
    X_train_flat, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
print('ANN оқыту аяқталды ✅')

## 6. Модель 2 — CNN (Конволюциялық нейрондық желі)

Екінші модель — суреттерді жіктеуге арналған CNN. ANN-ға қарағанда әлдеқайда күшті.

In [ ]:
# CNN модельді құру
model_cnn = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),

    # 1-ші конволюциялық блок
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # 2-ші конволюциялық блок
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    # Толық байланысты қабаттар
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
], name='CNN_Model')

model_cnn.summary()

In [ ]:
# Компиляция
model_cnn.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Оқыту
print('CNN моделін оқыту басталды...')
history_cnn = model_cnn.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
print('CNN оқыту аяқталды ✅')

## 7. Модельдерді бағалау және салыстыру

In [ ]:
# Test деректерінде бағалау
ann_loss, ann_acc = model_ann.evaluate(X_test_flat, y_test, verbose=0)
cnn_loss, cnn_acc = model_cnn.evaluate(X_test, y_test, verbose=0)

print('=== МОДЕЛЬДЕР САЛЫСТЫРУЫ ===')
print(f'ANN  — Дәлдік: {ann_acc*100:.2f}%,  Шығын: {ann_loss:.4f}')
print(f'CNN  — Дәлдік: {cnn_acc*100:.2f}%,  Шығын: {cnn_loss:.4f}')
print(f'\n🏆 Жеңімпаз: {"CNN" if cnn_acc > ann_acc else "ANN"}')

In [ ]:
# Оқыту графиктері
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ANN — Дәлдік
axes[0, 0].plot(history_ann.history['accuracy'], label='Train')
axes[0, 0].plot(history_ann.history['val_accuracy'], label='Validation')
axes[0, 0].set_title('ANN — Дәлдік (Accuracy)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()

# ANN — Шығын
axes[0, 1].plot(history_ann.history['loss'], label='Train')
axes[0, 1].plot(history_ann.history['val_loss'], label='Validation')
axes[0, 1].set_title('ANN — Шығын (Loss)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()

# CNN — Дәлдік
axes[1, 0].plot(history_cnn.history['accuracy'], label='Train')
axes[1, 0].plot(history_cnn.history['val_accuracy'], label='Validation')
axes[1, 0].set_title('CNN — Дәлдік (Accuracy)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()

# CNN — Шығын
axes[1, 1].plot(history_cnn.history['loss'], label='Train')
axes[1, 1].plot(history_cnn.history['val_loss'], label='Validation')
axes[1, 1].set_title('CNN — Шығын (Loss)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()

plt.suptitle('Модельдер оқыту нәтижелері', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# CNN Confusion Matrix
y_pred_cnn = np.argmax(model_cnn.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred_cnn)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('CNN — Confusion Matrix')
plt.xlabel('Болжанған санат')
plt.ylabel('Нақты санат')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print('=== CNN Classification Report ===')
print(classification_report(y_test, y_pred_cnn, target_names=class_names))

In [ ]:
# ANN vs CNN салыстыру бар диаграмма
models  = ['ANN', 'CNN']
accs    = [ann_acc * 100, cnn_acc * 100]
losses  = [ann_loss, cnn_loss]
colors  = ['#4C72B0', '#DD8452']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(models, accs, color=colors)
ax1.set_title('Дәлдік салыстыруы (%)')
ax1.set_ylabel('Accuracy (%)')
for i, v in enumerate(accs):
    ax1.text(i, v + 0.3, f'{v:.2f}%', ha='center', fontweight='bold')

ax2.bar(models, losses, color=colors)
ax2.set_title('Шығын салыстыруы (Loss)')
ax2.set_ylabel('Loss')
for i, v in enumerate(losses):
    ax2.text(i, v + 0.005, f'{v:.4f}', ha='center', fontweight='bold')

plt.suptitle('ANN vs CNN — Модельдер салыстыруы', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Қорытынды (Conclusion)

| Модель | Дәлдік | Шығын |
|--------|--------|-------|
| ANN    | ~50%   | ~1.4  |
| CNN    | ~70%   | ~0.9  |

**Нәтиже:**  
CNN моделі ANN-ға қарағанда суреттерді жіктеуде әлдеқайда жақсы нәтиже берді. Себебі CNN конволюциялық қабаттары арқылы суреттің ерекшеліктерін (edge, texture, shape) тиімді анықтайды.

**Жақсарту мүмкіндіктері:**
- Data Augmentation қолдану
- Epoch санын арттыру
- Batch Normalization қосу
- ResNet немесе VGG сияқты дайын архитектураларды қолдану
